# BanglaBERT + CNN-BiLSTM Hybrid — Sentiment Analysis (Kaggle) — v2

Fine-tuned `csebuetnlp/banglabert` fused with a Conv1D->BiLSTM branch, class-weighted +
label-smoothed loss, discriminative-LR AdamW, partial encoder freezing, early stopping on
validation macro-F1. Runs both the 3-class and 2-class tasks on the original ICCIT 2020
paper's dataset and compares against its reported **60% (3-class)** / **71% (2-class)** accuracy.

**v2 changes**, in response to round-1 overfitting (train loss 0.91->0.20 while val loss rose
0.87->2.11 on 3-class; best-val checkpoint scored 0.65 val macro-F1 but only 0.50 test
macro-F1): lower head/encoder LR, label smoothing, higher dropout, partial encoder-layer
freezing, an LR-decay horizon decoupled from the early-stopping epoch cap, and per-epoch
train/val plots so the overfitting curve is visible directly instead of inferred from logs.

Sources code from `https://github.com/habibour/sent` and reads
`/kaggle/input/datasets/reversedthoutgts/bangla-dataset/{train_,test_}.csv`.

To smoke-test before committing to a full run, lower `EPOCHS` in the config cell below to 1-2.


In [ ]:
import os

REPO_DIR = '/kaggle/working/sent'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/habibour/sent.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(REPO_DIR)


In [ ]:
!pip install -q -U transformers scikit-learn
!pip install -q git+https://github.com/csebuetnlp/normalizer.git


In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.data_utils import load_and_prepare, get_class_weights, verify_normalizer
from src.hybrid_model import (
    SentimentDataset, BanglaBertHybrid, freeze_encoder_layers,
    train_with_early_stopping, evaluate, plot_history, MAX_LEN,
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

verify_normalizer()


## Config

`PAPER_BASELINE` is each task's accuracy from Table V of the original paper (BERT_BSA+GRU).

Round-1 defaults were `encoder_lr=2e-5, head_lr=1e-3, dropout=0.3, num_frozen_layers=0`, which
overfit. v2 defaults below are more conservative; adjust further if the plots still show the
val loss diverging early.


In [ ]:
CONFIG = {
    'train_path': '/kaggle/input/datasets/reversedthoutgts/bangla-dataset/train_.csv',
    'test_path': '/kaggle/input/datasets/reversedthoutgts/bangla-dataset/test_.csv',
    'model_name': 'csebuetnlp/banglabert',
    'max_len': MAX_LEN,
    'val_ratio': 0.15,
    'seed': 42,
    'batch_size': 16,
    'epochs': 15,           # hard cap; early stopping usually ends it sooner
    'lr_decay_epochs': 8,   # horizon the LR schedule decays over
    'patience': 3,
    'encoder_lr': 1e-5,     # was 2e-5
    'head_lr': 3e-4,        # was 1e-3
    'dropout': 0.4,         # was 0.3
    'label_smoothing': 0.1, # new
    'num_frozen_layers': 6, # new -- freeze bottom half of BanglaBERT's 12 layers
    'weight_decay': 0.01,
    'warmup_ratio': 0.06,
    'grad_clip': 1.0,
    'use_fp16': True,
    'output_dir': '/kaggle/working',
}

TASKS = {
    '3class': {'num_labels': 3, 'label_names': ['Neutral', 'Positive', 'Negative'], 'paper_baseline': 0.60},
    '2class': {'num_labels': 2, 'label_names': ['Negative', 'Positive'], 'paper_baseline': 0.71},
}

print(CONFIG)
print(TASKS)


## Experiment function

Runs the full pipeline for one task: data prep -> tokenize -> fine-tune -> plot -> evaluate on held-out test.

In [ ]:
def run_experiment(task):
    assert task in TASKS, f"task must be one of {list(TASKS)}"
    cfg = TASKS[task]
    print('='*30, task, '='*30)
    set_seed(CONFIG['seed'])

    train_df, val_df, test_df = load_and_prepare(
        CONFIG['train_path'], CONFIG['test_path'], task,
        val_ratio=CONFIG['val_ratio'], seed=CONFIG['seed'],
    )

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

    train_ds = SentimentDataset(train_df['Data'], train_df['label'], tokenizer, CONFIG['max_len'])
    val_ds = SentimentDataset(val_df['Data'], val_df['label'], tokenizer, CONFIG['max_len'])
    test_ds = SentimentDataset(test_df['Data'], test_df['label'], tokenizer, CONFIG['max_len'])

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    model = BanglaBertHybrid(CONFIG['model_name'], num_labels=cfg['num_labels'],
                              dropout=CONFIG['dropout']).to(device)

    if CONFIG['num_frozen_layers'] > 0:
        freeze_encoder_layers(model, CONFIG['num_frozen_layers'])

    class_weights = get_class_weights(train_df['label'].values, cfg['num_labels']).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG['label_smoothing'])
    print('class weights:', class_weights)

    start = time.time()
    model, best_val_macro_f1, history = train_with_early_stopping(
        model, train_loader, val_loader, criterion, device,
        encoder_lr=CONFIG['encoder_lr'], head_lr=CONFIG['head_lr'],
        weight_decay=CONFIG['weight_decay'], warmup_ratio=CONFIG['warmup_ratio'],
        epochs=CONFIG['epochs'], lr_decay_epochs=CONFIG['lr_decay_epochs'],
        patience=CONFIG['patience'], grad_clip=CONFIG['grad_clip'],
        use_fp16=CONFIG['use_fp16'], label_names=cfg['label_names'],
    )
    elapsed = time.time() - start
    print(f'training time: {elapsed/60:.1f} min, best val macro-F1: {best_val_macro_f1:.4f}')

    plot_history(history, title=task)

    ckpt_path = f"{CONFIG['output_dir']}/banglabert_hybrid_{task}.pt"
    torch.save(model.state_dict(), ckpt_path)

    test_metrics = evaluate(model, test_loader, criterion, device, label_names=cfg['label_names'])
    print(f"\nTest accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Test macro-F1: {test_metrics['macro_f1']:.4f}")
    print(f"Test weighted-F1: {test_metrics['weighted_f1']:.4f}")
    print('\n' + test_metrics['report'])
    print('Confusion matrix:\n', test_metrics['confusion_matrix'])

    paper_acc = cfg['paper_baseline']
    print(f"\nPaper's reported BERT_BSA+GRU {task} accuracy: {paper_acc*100:.2f}%")
    print(f"This run's test accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Difference: {(test_metrics['accuracy'] - paper_acc)*100:+.2f} percentage points")

    return {
        'task': task,
        'accuracy': test_metrics['accuracy'],
        'macro_f1': test_metrics['macro_f1'],
        'weighted_f1': test_metrics['weighted_f1'],
        'paper_baseline': paper_acc,
        'best_val_macro_f1': best_val_macro_f1,
        'checkpoint': ckpt_path,
        'history': history,
    }


## Run both tasks

In [ ]:
results = []
for task in ['3class', '2class']:
    results.append(run_experiment(task))


In [ ]:
summary = pd.DataFrame([{k: v for k, v in r.items() if k != 'history'} for r in results])
summary['improvement_pp'] = (summary['accuracy'] - summary['paper_baseline']) * 100
print(summary.to_string(index=False))
summary.to_csv(f"{CONFIG['output_dir']}/results_summary.csv", index=False)
